In [8]:
# Import library
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [9]:
df = pd.read_csv('../data/raw/online_retail_II.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [10]:
# Cek data type dan missing value jika ada
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [11]:
# Cek statistik dasar
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


In [ ]:
# Cek jumlah missing value
print(df.isnull().sum())
print(f"Persentase missing values:")
print(round(df.isnull().sum() / len(df) * 100, 2))

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Persentase missing values:
Invoice         0.00
StockCode       0.00
Description     0.41
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    22.77
Country         0.00
dtype: float64


In [13]:
# Cek duplikat
print(f"Jumlah duplikat: {df.duplicated().sum()}")

Jumlah duplikat: 34335


In [14]:
# Cek transaksi cancelled (Invoice prefix 'C')
cancelled = df[df['Invoice'].astype(str).str.startswith('C')]
print(f"Jumlah transaksi cancelled: {len(cancelled)}")

# Cek Price = 0
print(f"Jumlah Price = 0: {len(df[df['Price'] == 0])}")

# Cek Price negatif
print(f"Jumlah Price negatif: {len(df[df['Price'] < 0])}")

# Cek Quantity negatif
print(f"Jumlah Quantity negatif: {len(df[df['Quantity'] < 0])}")

Jumlah transaksi cancelled: 19494
Jumlah Price = 0: 6202
Jumlah Price negatif: 5
Jumlah Quantity negatif: 22950


In [15]:
# Pisahkan transaksi cancelled sebagai dataframe terpisah
df_cancelled = df[df['Invoice'].astype(str).str.startswith('C')].copy()
print(f"Data cancelled: {df_cancelled.shape}")

# Simpan untuk analisis cancellation nanti
df_cancelled.to_csv('../data/cleaned/cancelled_transactions.csv', index=False)

# Hapus dari dataframe utama
df = df[~df['Invoice'].astype(str).str.startswith('C')]
print(f"Data setelah hapus cancelled: {df.shape}")

Data cancelled: (19494, 8)
Data setelah hapus cancelled: (1047877, 8)


In [16]:
# Pisahkan transaksi retur sebagai dataframe terpisah
df_returns = df[df['Quantity'] < 0].copy()
print(f"Data returns: {df_returns.shape}")

# Simpan untuk analisis retur nanti
df_returns.to_csv('../data/cleaned/return_transactions.csv', index=False)

# Hapus dari dataframe utama
df = df[df['Quantity'] > 0]
print(f"Data setelah hapus returns: {df.shape}")

Data returns: (3457, 8)
Data setelah hapus returns: (1044420, 8)


In [ ]:
print(f"Shape sebelum drop missing: {df.shape}")

# Drop missing Customer ID dan Description
df = df.dropna(subset=['Customer ID', 'Description'])

print(f"Shape setelah drop missing: {df.shape}")
print(f"Missing values sekarang:")
print(df.isnull().sum())

Shape sebelum drop missing: (1044420, 8)
Shape setelah drop missing: (805620, 8)

Missing values sekarang:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64


In [18]:
print(f"Shape sebelum drop duplikat: {df.shape}")

df = df.drop_duplicates()

print(f"Shape setelah drop duplikat: {df.shape}")

Shape sebelum drop duplikat: (805620, 8)
Shape setelah drop duplikat: (779495, 8)


In [19]:
print(f"Shape sebelum filter price: {df.shape}")

# Drop Price = 0 dan Price negatif
df = df[df['Price'] > 0]

print(f"Shape setelah filter price: {df.shape}")

Shape sebelum filter price: (779495, 8)
Shape setelah filter price: (779425, 8)


In [ ]:
# Convert InvoiceDate dari string ke datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Convert Customer ID dari float ke integer ke string
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)

# Verifikasi
print(df.dtypes)
print(f"Sample InvoiceDate: {df['InvoiceDate'].iloc[0]}")
print(f"Sample Customer ID: {df['Customer ID'].iloc[0]}")

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
dtype: object

Sample InvoiceDate: 2009-12-01 07:45:00
Sample Customer ID: 13085


In [21]:
# TotalPrice per baris transaksi
df['TotalPrice'] = df['Quantity'] * df['Price']

# Ekstrak komponen waktu
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()

# Verifikasi
print(df[['InvoiceDate', 'Year', 'Month', 'DayOfWeek', 'TotalPrice']].head())

          InvoiceDate  Year  Month DayOfWeek  TotalPrice
0 2009-12-01 07:45:00  2009     12   Tuesday        83.4
1 2009-12-01 07:45:00  2009     12   Tuesday        81.0
2 2009-12-01 07:45:00  2009     12   Tuesday        81.0
3 2009-12-01 07:45:00  2009     12   Tuesday       100.8
4 2009-12-01 07:45:00  2009     12   Tuesday        30.0


In [ ]:
print("=== FINAL DATA SUMMARY ===")
print(f"Shape: {df.shape}")
print(f"Missing values:{df.isnull().sum()}")
print(f"Data types:{df.dtypes}")
print(f"Sample data:")
df.head()

=== FINAL DATA SUMMARY ===
Shape: (779425, 12)

Missing values:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
TotalPrice     0
Year           0
Month          0
DayOfWeek      0
dtype: int64

Data types:
Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
TotalPrice            float64
Year                    int32
Month                   int32
DayOfWeek                 str
dtype: object

Sample data:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice,Year,Month,DayOfWeek
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,2009,12,Tuesday
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009,12,Tuesday
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009,12,Tuesday
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,2009,12,Tuesday
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,2009,12,Tuesday


In [25]:
# Simpan cleaned data
df.to_csv('../data/cleaned/cleaned_data.csv', index=False)

In [ ]:
import sqlite3

# Buat koneksi ke SQLite
conn = sqlite3.connect('../data/cleaned/retail.db')

# Simpan dataframe utama
df.to_sql('retail', conn, if_exists='replace', index=False)

# Simpan juga cancelled & returns untuk analisis nanti
df_cancelled.to_sql('cancelled', conn, if_exists='replace', index=False)
df_returns.to_sql('returns', conn, if_exists='replace', index=False)

# Verifikasi
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM retail")
print(f"Tabel retail: {cursor.fetchone()[0]} baris")

cursor.execute("SELECT COUNT(*) FROM cancelled")
print(f"Tabel cancelled: {cursor.fetchone()[0]} baris")

cursor.execute("SELECT COUNT(*) FROM returns")
print(f"Tabel returns: {cursor.fetchone()[0]} baris")

conn.close()
print("retail.db berhasil disimpan.")

✅ Tabel retail: 779425 baris
✅ Tabel cancelled: 19494 baris
✅ Tabel returns: 3457 baris

✅ retail.db berhasil disimpan!
